# Huấn Luyện Các Mô Hình Tích Hợp (AdaBoost Ensembles)

Notebook này thực hiện huấn luyện và đánh giá mô hình trên toàn bộ 10 tập dữ liệu với 5 độ đo (Accuracy, Precision, Recall, F-Score, ROC-AUC) sử dụng 10-Fold CV và SMOTE.


In [1]:
import sys
import os
import glob
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.data_loader import load_arff_dataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')

In [2]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

def get_models(random_state=42):
    base_estimators = {
        'RF': RandomForestClassifier(random_state=random_state, n_jobs=-1),
        'DS': DecisionTreeClassifier(random_state=random_state),
        'SVM': SVC(kernel='linear', probability=True, random_state=random_state, max_iter=2000),
        'LR': LogisticRegression(max_iter=1500, random_state=random_state, n_jobs=-1)
    }
    
    adaboost_models = {}
    for name, clf in base_estimators.items():
        adaboost_models[f'AdaBoost_{name}'] = AdaBoostClassifier(
            estimator=clf,
            random_state=random_state
        )
    return adaboost_models

In [3]:
def run_experiment_on_dataset(df, target_col='Defective', random_state=42):
    df = df.dropna()
    X = df.drop(columns=[target_col]).values
    y = df[target_col].map({'Defective': 1, 'Clean': 0}).values
    
    kf = KFold(n_splits=10, shuffle=True, random_state=random_state)
    models_dict = get_models()
    
    results_accumulator = {
        name: {'Accuracy': [], 'Precision': [], 'Recall': [], 'F-Score': [], 'ROC-AUC': []}
        for name in models_dict.keys()
    }
    
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        for model_name, model in models_dict.items():
            pipeline = ImbPipeline([
                ('scaler', StandardScaler()),
                ('smote', SMOTE(random_state=random_state)),
                ('model', model)
            ])
            
            try:
                pipeline.fit(X_train, y_train)
                predictions = pipeline.predict(X_test)
                if hasattr(pipeline.named_steps['model'], "predict_proba"):
                    probabilities = pipeline.predict_proba(X_test)[:, 1]
                else:
                    probabilities = pipeline.decision_function(X_test)
                
                acc = accuracy_score(y_test, predictions)
                prec = precision_score(y_test, predictions, average='weighted', zero_division=0)
                rec = recall_score(y_test, predictions, average='weighted', zero_division=0)
                f1 = f1_score(y_test, predictions, average='weighted', zero_division=0)
                try:
                    auc = roc_auc_score(y_test, probabilities, average='weighted')
                except ValueError:
                    auc = np.nan
                    
                results_accumulator[model_name]['Accuracy'].append(acc)
                results_accumulator[model_name]['Precision'].append(prec)
                results_accumulator[model_name]['Recall'].append(rec)
                results_accumulator[model_name]['F-Score'].append(f1)
                results_accumulator[model_name]['ROC-AUC'].append(auc)
            except Exception as e:
                pass
                
    final_report = {}
    for model_name, metrics in results_accumulator.items():
        final_report[model_name] = {
            'Accuracy': np.round(np.mean(metrics['Accuracy']), 3) if metrics['Accuracy'] else 0.0,
            'Precision': np.round(np.mean(metrics['Precision']), 3) if metrics['Precision'] else 0.0,
            'Recall': np.round(np.mean(metrics['Recall']), 3) if metrics['Recall'] else 0.0,
            'F-Score': np.round(np.mean(metrics['F-Score']), 3) if metrics['F-Score'] else 0.0,
            'ROC-AUC': np.round(np.nanmean(metrics['ROC-AUC']), 3) if metrics['ROC-AUC'] else 0.0
        }
    return final_report

In [4]:
dataset_dir = '../datasets'
arff_files = glob.glob(os.path.join(dataset_dir, '*.arff'))

all_results = []

print(f"Tìm thấy {len(arff_files)} tập dữ liệu. Bắt đầu huấn luyện...")

for file_path in sorted(arff_files):
    ds_name = os.path.splitext(os.path.basename(file_path))[0]
    print(f"\n>>> Đang xử lý tập dữ liệu: {ds_name}...")
    df, target_col = load_arff_dataset(file_path)
    report = run_experiment_on_dataset(df, target_col)
    
    for model_name, metrics in report.items():
        row = {'Dataset': ds_name, 'Model': model_name}
        row.update(metrics)
        all_results.append(row)
        
results_df = pd.DataFrame(all_results)
results_df = results_df[['Dataset', 'Model', 'Accuracy', 'Precision', 'Recall', 'F-Score', 'ROC-AUC']]

print("\n================ HOÀN TẤT THỰC NGHIỆM ================")
display(results_df)

# Lưu báo cáo
output_file = 'report_AdaBoost.csv'
results_df.to_csv(output_file, index=False)
print(f"\nKết quả đã được lưu thành công vào: {output_file}")

Tìm thấy 10 tập dữ liệu. Bắt đầu huấn luyện...

>>> Đang xử lý tập dữ liệu: JM1...

>>> Đang xử lý tập dữ liệu: KC3...

>>> Đang xử lý tập dữ liệu: MC1...

>>> Đang xử lý tập dữ liệu: MC2...

>>> Đang xử lý tập dữ liệu: MW1...

>>> Đang xử lý tập dữ liệu: PC1...

>>> Đang xử lý tập dữ liệu: PC2...

>>> Đang xử lý tập dữ liệu: PC3...

>>> Đang xử lý tập dữ liệu: PC4...

>>> Đang xử lý tập dữ liệu: PC5...

================ HOÀN TẤT THỰC NGHIỆM ================


,Dataset,Model,Accuracy,Precision,Recall,F-Score,ROC-AUC
0,JM1,AdaBoost_RF,0.781,0.764,0.781,0.770,0.630
1,JM1,AdaBoost_DS,0.699,0.722,0.699,0.709,0.582
2,JM1,AdaBoost_SVM,0.000,0.000,0.000,0.000,0.000
3,JM1,AdaBoost_LR,0.718,0.754,0.718,0.732,0.686
4,KC3,AdaBoost_RF,0.804,0.812,0.804,0.799,0.672
5,KC3,AdaBoost_DS,0.789,0.820,0.789,0.795,0.704
6,KC3,AdaBoost_SVM,0.649,0.750,0.649,0.669,0.623
7,KC3,AdaBoost_LR,0.717,0.783,0.717,0.737,0.739
8,MC1,AdaBoost_RF,0.979,0.976,0.979,0.977,0.625
9,MC1,AdaBoost_DS,0.956,0.971,0.956,0.963,0.595



Kết quả đã được lưu thành công vào: report_AdaBoost.csv
